# Priors

This example shows how Bayesian priors change a BKT fit, using a small dataset where the priors matter. It covers:

1. The logit scale that priors are specified on.
2. Default, informative and uniform priors with `StandardPriors`.
3. Different priors per KC with a dict.
4. Per group priors with `MultiPriors`.

See the [Priors](../guide/priors.rst) guide page for the full reference.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

## Simulate a small dataset

With only 15 students and 10 problems per student, the data alone pins down the parameters loosely, so the priors have a visible effect.

In [2]:
from stanbkt.utils import sim_simple_BKT

true_params = {"prior": 0.2, "learn": 0.15, "forget": 0.02, "guess": 0.2, "slip": 0.1}
data_df = sim_simple_BKT(n_students=15, n_problems=10, n_kcs=1, rng_seed=7, **true_params)
data_df.head()

,student_id,problem_id,correct,timestamp,kc_id
0,stu_0,prob_0,0,2024-01-01 00:00:00,kc_0
1,stu_0,prob_1,0,2024-01-01 00:01:00,kc_0
2,stu_0,prob_2,1,2024-01-01 00:02:00,kc_0
3,stu_0,prob_3,1,2024-01-01 00:03:00,kc_0
4,stu_0,prob_4,1,2024-01-01 00:04:00,kc_0


## The logit scale

Priors are normal distributions on the unconstrained scale of each parameter. `pi_know`, `learn` and `forget` use the logit, and `guess` and `slip` use the _half-logit_ (`guess = 0.5 * inv_logit(x)`), which caps them at 0.5. To center a prior on a probability, convert it first.

In [3]:
def logit(p):
    return np.log(p / (1 - p))


def inv_logit(x):
    return 1 / (1 + np.exp(-x))


# a learn rate of 0.15 on the logit scale, and a guess of 0.2 on the half-logit scale
print(logit(0.15), logit(0.2 / 0.5))
# the default prior means, back on the probability scale
print("pi_know", inv_logit(-2.0), "learn", inv_logit(0.0), "guess", 0.5 * inv_logit(-1.0))

-1.7346010553881064 -0.4054651081081643
pi_know 0.11920292202211755 learn 0.5 guess 0.13447071068499755


## Fit with three sets of priors

- **Default**: `StandardPriors()`, weakly informative normal priors with a standard deviation of 5.
- **Informative**: narrow priors centered near plausible values, here close to the truth.
- **Uniform**: `StandardPriors(use_defaults=False)` sets every prior to `None`, a flat prior on the logit scale.

Each model is fitted separately so we can compare their posteriors.

In [4]:
from stanbkt.fits import MCMCFitOptions
from stanbkt.models import StandardBKT, StandardPriors
from stanbkt.utils import VerbosityLevel

fit_opts = MCMCFitOptions(seed=1234, iter_warmup=500, iter_sampling=500, show_progress=False)

priors = {
    "default": StandardPriors(),
    "informative": StandardPriors(
        learn_mu=logit(0.15), learn_std=0.5,
        guess_mu=logit(0.2 / 0.5), guess_std=0.5,
        slip_mu=logit(0.1 / 0.5), slip_std=0.5,
    ),
    "uniform": StandardPriors(use_defaults=False),
}

models = {}
for name, prior in priors.items():
    models[name] = StandardBKT(verbose=VerbosityLevel.WARN)
    models[name].fit(data_df, priors=prior, stan_fit_options=fit_opts)

23:57:11 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 41 divergent transitions (8.2%)
	Chain 1 had 441 iterations at max treedepth (88.2%)
	Chain 2 had 72 divergent transitions (14.4%)
	Chain 2 had 410 iterations at max treedepth (82.0%)
	Chain 3 had 35 divergent transitions (7.0%)
	Chain 3 had 438 iterations at max treedepth (87.6%)
	Chain 4 had 69 divergent transitions (13.8%)
	Chain 4 had 397 iterations at max treedepth (79.4%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


The summary has one row per parameter. We compare the probability scale parameters (`learn`, `guess`, `slip`, `pi_know`) across the three fits.

In [5]:
params = ["pi_know", "learn", "guess", "slip"]
rows = []
for name, model in models.items():
    summary = model.summary().loc["kc_0"]
    for param in params:
        row = summary.loc[param].iloc[0]
        rows.append({"priors": name, "parameter": param, "mean": row["Mean"],
                     "2.5%": row["2.5%"], "97.5%": row["97.5%"], "R_hat": row["R_hat"]})
comparison = pd.DataFrame(rows).pivot(index="parameter", columns="priors")
comparison.round(3)

mean                        2.5%                       97.5%  \
priors    default informative uniform default informative uniform default   
parameter                                                                   
guess       0.241       0.216    0.50   0.104       0.133    0.50   0.398   
learn       0.119       0.129    0.08   0.041       0.068    0.03   0.228   
pi_know     0.150       0.167    0.00   0.000       0.000    0.00   0.482   
slip        0.067       0.090    0.00   0.003       0.047    0.00   0.152   

                                R_hat                      
priors    informative uniform default informative uniform  
parameter                                                  
guess           0.302   0.500   1.014       1.000     NaN  
learn           0.207   0.147   1.012       1.002   1.004  
pi_know         0.495   0.000   1.003       1.000     NaN  
slip            0.148   0.000   1.003       1.004     NaN

In [6]:
true_params

{'prior': 0.2, 'learn': 0.15, 'forget': 0.02, 'guess': 0.2, 'slip': 0.1}

The informative priors give the narrowest intervals because they add information the small dataset does not have.

The uniform priors show why priors matter in BKT. Without them, this small dataset is explained just as well by a degenerate model where `guess` sits at its 0.5 cap and `slip` and `pi_know` collapse to 0, so every correct answer is a guess and learning does the rest. The chains get stuck at these boundaries, which is why `R_hat` is `NaN`. The weakly informative defaults are enough to keep the fit away from them. Always check `R_hat` and the effective sample size, especially with flat priors (see [Summaries and Diagnostics](../guide/summaries.rst)).

## Different priors per KC

A single priors object applies to every KC. To set priors per KC, pass a dict keyed by KC ID. KCs not in the dict use the default priors.

In [7]:
two_kc_df = sim_simple_BKT(n_students=15, n_problems=10, n_kcs=2, rng_seed=8,
                           learn=[0.1, 0.3], guess=0.2, slip=0.1)

per_kc_priors = {
    "kc_0": StandardPriors(learn_mu=logit(0.1), learn_std=0.5),
    "kc_1": StandardPriors(learn_mu=logit(0.3), learn_std=0.5),
}
per_kc_model = StandardBKT(verbose=VerbosityLevel.WARN)
per_kc_model.fit(two_kc_df, priors=per_kc_priors, stan_fit_options=fit_opts)
per_kc_model.summary().xs("learn", level="parameter")[["Mean", "2.5%", "97.5%"]]

,,,Mean,2.5%,97.5%
kc_id,axis,label,,,
kc_0,group_id,1,0.108429,0.043847,0.213640
kc_1,group_id,1,0.226586,0.126350,0.385585


## Per group priors with MultiPriors

`MultiBKT` takes `MultiPriors`. Each field is either a scalar shared by every group or a list with one value per group. Groups are indexed per KC in order of first appearance, with students in natural sort order, so check the group labels in `summary()`.

In [8]:
from stanbkt.models import MultiBKT, MultiPriors
from stanbkt.utils import sim_grouped_BKT

grouped_df = sim_grouped_BKT(n_students=30, n_problems=10, n_kcs=1, n_groups=2, rng_seed=9,
                             learn=[0.05, 0.3], guess=0.2, slip=0.1)

multi_priors = MultiPriors(learn_mu=[logit(0.05), logit(0.3)], learn_std=[0.5, 0.5])
multi_model = MultiBKT(verbose=VerbosityLevel.WARN)
multi_model.fit(grouped_df, priors=multi_priors, stan_fit_options=fit_opts)
multi_model.summary().loc["kc_0"].loc["learn"][["Mean", "2.5%", "97.5%"]]

Mean      2.5%     97.5%
axis     label                                
group_id group_0  0.052656  0.021917  0.100204
         group_1  0.274769  0.159956  0.422480

A list must have one entry per group in every KC it is applied to. If KCs have different groups, use a dict of `MultiPriors` per KC.